# site data
flux, radiation, aerosol and smoke time series for each site from the raw downloads, then cloud screening, smoke and clean reference samples, growing season, crop years and the temperature and vpd filter. writes the samples used by the modelling notebooks and the aod series of figure 1

In [1]:
from pathlib import Path
import glob, io, re, zipfile
import numpy as np, pandas as pd, shapefile
from shapely.geometry import shape, Point

raw, data = Path("../data/raw"), Path("data")
data.mkdir(exist_ok=True)
# lat, lon, utc offset of local standard time, aeronet sites, phenocam camera
sites = {
    "US-Bi2": (38.1091, -121.5351, -8, ["Sacramento_River", "Sacramento_CARB", "Grizzly_Bay", "Modesto", "NASA_Ames",
                                        "Turlock_CA_USA"], "bouldincorn_AG_1000"),
    "US-Ne1": (41.1651, -96.4766, -6, ["NEON_KONZ", "Ames", "NEON_UKFS", "Sioux_Falls"], "mead1_AG_1000"),
    "US-Ne2": (41.1649, -96.4701, -6, ["NEON_KONZ", "Ames", "NEON_UKFS", "Sioux_Falls"], "mead2_AG_1000"),
    "US-Ne3": (41.1797, -96.4397, -6, ["NEON_KONZ", "Ames", "NEON_UKFS", "Sioux_Falls"], "mead3_AG_1000"),
    "US-xUN": (46.2339, -89.5373, -6, ["NEON_UNDE"], "NEON.D05.UNDE.DP1.00033_DB_1000"),
    "US-UMB": (45.5598, -84.7138, -5, ["Granite_Island", "South_Greenbay"], "umichbiological_DB_2000"),
    "US-MMS": (39.3232, -86.4131, -5, ["BONDVILLE", "Dayton"], "morganmonroe_DB_1000"),
    "US-Ha1": (42.5378, -72.1715, -5, ["NEON_Harvard"], "harvard_DB_1000")}
maize = ["US-Ne1", "US-Ne2", "US-Ne3"]
broadband = ["US-xUN"]  # diffuse fraction from a shortwave sensor
drivers = ["PAR", "DF", "COSZ", "VPD", "TA", "SWC"]
cols = ["date", "hour", "GPP"] + drivers + ["AOD550", "AE", "ssa_sw", "g_sw", "is_smoke", "is_bg"]

In [2]:
def first(d, *names):
    return next((d[c] for c in names if c in d and d[c].notna().any()), pd.Series(np.nan, index=d.index))


def flux(site):
    # ameriflux fluxnet half hourly or hourly file; par from the base product when fluxnet has none (mead)
    with zipfile.ZipFile(sorted(raw.glob(f"ameriflux/AMF_{site}_FLUXNET_*.zip"))[-1]) as z:
        name = next(n for n in z.namelist() if re.search(r"(FULLSET|FLUXMET)_H[HR]_", n))
        want = ["TIMESTAMP_START", "GPP_NT_VUT_REF", "GPP_NT_VUT_MEAN", "GPP_DT_VUT_REF", "NEE_VUT_REF_QC", "SW_IN",
                "SW_IN_F_MDS", "SW_IN_F_MDS_QC", "SW_IN_POT", "SW_DIF", "PPFD_IN", "PPFD_DIF", "VPD_F_MDS", "VPD_F",
                "TA_F_MDS", "TA_F", "SWC_F_MDS_1", "SWC_F_MDS_2"]
        f = pd.read_csv(z.open(name), usecols=lambda c: c in want, na_values=[-9999], dtype={"TIMESTAMP_START": str})
    f["TIMESTAMP_START"] = pd.to_datetime(f.TIMESTAMP_START, format="%Y%m%d%H%M")
    missing = [v for v in ("PPFD_IN", "PPFD_DIF") if v not in f or f[v].isna().all()]
    base = sorted(raw.glob(f"ameriflux/AMF_{site}_BASE-BADM_*.zip"))
    if missing and base:
        with zipfile.ZipFile(base[-1]) as z:
            name = next(n for n in z.namelist() if re.search(r"_BASE_H[HR]_", n))
            b = pd.read_csv(z.open(name), skiprows=2, na_values=[-9999], dtype={"TIMESTAMP_START": str}, low_memory=False)
        b["TIMESTAMP_START"] = pd.to_datetime(b.TIMESTAMP_START, format="%Y%m%d%H%M")
        for v in missing:
            c = next((c for c in (f"{v}_1_1_1", v, f"{v}_PI_F_1_1_1", f"{v}_PI_F") if c in b and b[c].notna().mean() > 0.2), None)
            if c:
                f = f.drop(columns=v, errors="ignore").merge(b[["TIMESTAMP_START", c]].rename(columns={c: v}), how="left")
    swc = ["SWC_F_MDS_2"] if site == "US-Ha1" else ["SWC_F_MDS_1", "SWC_F_MDS_2"]
    return pd.DataFrame({"TIMESTAMP_START": f.TIMESTAMP_START,
                         "GPP": first(f, "GPP_NT_VUT_REF", "GPP_NT_VUT_MEAN", "GPP_DT_VUT_REF"),
                         "NEE_QC": first(f, "NEE_VUT_REF_QC"),
                         "SW_IN": first(f, "SW_IN", "SW_IN_F_MDS").where(first(f, "SW_IN_F_MDS_QC").fillna(0) == 0),
                         "SW_IN_POT": first(f, "SW_IN_POT"), "SW_DIF": first(f, "SW_DIF"),
                         "PPFD_IN": first(f, "PPFD_IN"), "PPFD_DIF": first(f, "PPFD_DIF"),
                         "VPD": first(f, "VPD_F_MDS", "VPD_F"), "TA": first(f, "TA_F_MDS", "TA_F"), "SWC": first(f, *swc)})


def solar_zenith(t, lat, lon, utc):
    # noaa solar position, t in local standard time
    t = pd.DatetimeIndex(t) - pd.Timedelta(hours=utc)
    doy, hr = t.dayofyear.to_numpy(), (t.hour + t.minute / 60).to_numpy()
    g = 2 * np.pi / 365 * (doy - 1 + (hr - 12) / 24)
    eqt = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g) - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    dec = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g) + 0.000907 * np.sin(2 * g)
           - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    ha = np.radians((hr * 60 + eqt + 4 * lon) / 4 - 180)
    la = np.radians(lat)
    return np.degrees(np.arccos(np.clip(np.sin(la) * np.sin(dec) + np.cos(la) * np.cos(dec) * np.cos(ha), -1, 1)))

In [3]:
def read_aeronet(names, kind):
    out = []
    for s in names:
        for f in sorted(glob.glob(str(raw / "aeronet" / f"{s}_*_{kind}.csv"))):
            if Path(f).stat().st_size == 0:
                continue
            lines = open(f, encoding="utf-8", errors="replace").readlines()
            head = next(i for i, l in enumerate(lines) if l.startswith("AERONET_Site,"))
            a = pd.read_csv(f, skiprows=head, na_values=[-999], encoding_errors="replace", low_memory=False)
            a.columns = [c.strip().replace("Date_(", "Date(").replace("Time_(", "Time(") for c in a.columns]
            utc = pd.to_datetime(a["Date(dd:mm:yyyy)"] + " " + a["Time(hh:mm:ss)"], format="%d:%m:%Y %H:%M:%S")
            a = pd.concat([a, utc.rename("utc")], axis=1)
            out.append(a[a.AERONET_Site == s])
    return pd.concat(out, ignore_index=True)


def aod(names, utc):
    # aod at 550 nm from 500 nm and the 440 to 870 nm angstrom exponent; 30 minute means per aeronet site, then over sites
    a = read_aeronet(names, "aod15")[["AERONET_Site", "utc", "AOD_500nm", "440-870_Angstrom_Exponent"]].copy()
    a["AE"] = a["440-870_Angstrom_Exponent"]
    a["AOD550"] = a["AOD_500nm"] * (550 / 500) ** (-a.AE)
    a = a.dropna(subset=["AOD550"])
    a["TIMESTAMP_START"] = (a.utc + pd.Timedelta(hours=utc)).dt.floor("30min")
    g = a.groupby(["TIMESTAMP_START", "AERONET_Site"]).agg(AOD550=("AOD550", "mean"), AE=("AE", "mean"), n=("AOD550", "size"))
    return g.groupby("TIMESTAMP_START").agg(AOD550=("AOD550", "mean"), AE=("AE", "mean"), n_obs=("n", "sum")).reset_index()


def inversions(names, utc):
    # shortwave single scattering albedo and asymmetry: inversion values at 440, 675, 870, 1020 nm weighted by top of
    # atmosphere solar energy in each band times (wavelength / 550) ^ -1.5, daily median
    wl = np.array([440, 675, 870, 1020])
    w = np.array([520.0, 330.0, 170.0, 290.0]) * (wl / 550.0) ** -1.5
    sc, gc = [f"Single_Scattering_Albedo[{x}nm]" for x in wl], [f"Asymmetry_Factor-Total[{x}nm]" for x in wl]
    key = ["AERONET_Site", "utc"]
    m = read_aeronet(names, "inv_SSA")[key + sc].merge(read_aeronet(names, "inv_ASY")[key + gc], on=key)
    om, gg = m[sc].to_numpy(), m[gc].to_numpy()
    m["ssa_sw"] = (om * w).sum(1) / w.sum()
    m["g_sw"] = (gg * om * w).sum(1) / (om * w).sum(1)
    m["date"] = (m.utc + pd.Timedelta(hours=utc)).dt.normalize()
    return m.groupby("date")[["ssa_sw", "g_sw"]].median()

In [4]:
# noaa hms analyst smoke polygons over each tower, may to september 2016 to 2025
points = {s: Point(v[1], v[0]) for s, v in sites.items()}
rows = []
for day in pd.date_range("2016-05-01", "2025-09-30"):
    f = raw / "hms" / f"hms_smoke{day:%Y%m%d}.zip"
    if day.month not in range(5, 10) or not f.exists():
        continue
    with zipfile.ZipFile(f) as z:
        shp = next(n for n in z.namelist() if n.endswith(".shp"))
        rd = shapefile.Reader(shp=io.BytesIO(z.read(shp)), dbf=io.BytesIO(z.read(shp[:-4] + ".dbf")),
                              shx=io.BytesIO(z.read(shp[:-4] + ".shx")))
        polys = [shape(r.shape.__geo_interface__) for r in rd.iterShapeRecords()]
    for s, pt in points.items():
        hit = any(p.contains(pt) for p in polys if p.is_valid) or any(p.buffer(0).contains(pt) for p in polys if not p.is_valid)
        rows.append({"date": day, "site": s, "hms_smoke": int(hit)})
hms = pd.DataFrame(rows).set_index(["site", "date"]).hms_smoke

In [5]:
def merged(site):
    lat, lon, utc, aeronet, cam = sites[site]
    d = flux(site)
    step = d.TIMESTAMP_START.iloc[1] - d.TIMESTAMP_START.iloc[0]
    a = aod(aeronet, utc)
    if step == pd.Timedelta("60min"):
        a = a.groupby(a.TIMESTAMP_START.dt.floor("60min")).agg(AOD550=("AOD550", "mean"), AE=("AE", "mean"),
                                                               n_obs=("n_obs", "sum")).reset_index()
    d = d.merge(a, on="TIMESTAMP_START", how="left")
    d["sza"] = solar_zenith(d.TIMESTAMP_START + step / 2, lat, lon, utc)
    d["date"] = d.TIMESTAMP_START.dt.normalize()
    d["hms_smoke"] = d.date.map(hms[site]).fillna(0).astype(int)
    # days without an inversion take the median of smoke days (daily median aod above 0.3), clean days (at most 0.15),
    # or all inversion days
    inv = inversions(aeronet, utc)
    day_aod = d.groupby("date").AOD550.median()
    daod = d.date.map(day_aod)
    inv_aod = day_aod.reindex(inv.index)
    for c in ("ssa_sw", "g_sw"):
        fill = np.where(daod > 0.3, inv.loc[inv_aod > 0.3, c].median(),
                        np.where(daod <= 0.15, inv.loc[inv_aod <= 0.15, c].median(), inv.loc[inv.index.isin(day_aod.index), c].median()))
        d[c] = d.date.map(inv[c]).fillna(pd.Series(fill, index=d.index))
    return d.sort_values("TIMESTAMP_START").reset_index(drop=True)

In [6]:
def greenness(site):
    # phenocam canopy greenness as the fraction of the way from that year's least green to its most green
    # (2nd and 98th percentiles), after removing outliers, filling gaps up to 7 days and a 3 day running median
    g = pd.read_csv(raw / "phenocam" / f"{sites[site][4]}_1day.csv", comment="#", parse_dates=["date"])
    g = g[g.outlierflag_gcc_90.fillna(0) == 0].set_index("date").gcc_90.asfreq("D")
    g = g.interpolate(limit=7).rolling(3, center=True, min_periods=1).median()
    lo, hi = (g.groupby(g.index.year).transform("quantile", q) for q in (0.02, 0.98))
    return ((g - lo) / (hi - lo)).rename("green")


cdl = pd.read_csv(raw / "cdl_crop_by_year.csv", index_col="year")


def sample(site, d, hemes=True):
    t = d.TIMESTAMP_START
    d = d.assign(hour=t.dt.hour + t.dt.minute / 60, PAR=d.PPFD_IN, COSZ=np.cos(np.radians(d.sza)))
    df = d.SW_DIF / d.SW_IN if site in broadband else d.PPFD_DIF / d.PPFD_IN
    d["DF"] = df.where(df.between(0, 1))

    # cloud free: daytime, clearness index at least 0.45, steady over neighbouring steps, aeronet observation in the
    # step, and shortwave not above 1.05 times the clear sky envelope for that time of day and day of year
    day = d.SW_IN_POT > 50
    kt = (d.SW_IN / d.SW_IN_POT).where(day)
    dk = 0.03 if (t.iloc[1] - t.iloc[0]).total_seconds() < 2700 else 0.04
    steady = ((kt - kt.shift(1)).abs() < dk) & ((kt - kt.shift(-1)).abs() < dk)
    doy, slot = t.dt.dayofyear, t.dt.hour * 2 + t.dt.minute // 30
    env = pd.Series(np.nan, index=d.index)
    for s in slot.unique():
        m = slot == s
        clim = d.SW_IN.where(day)[m].groupby(doy[m]).quantile(0.98).reindex(range(1, 367))
        clim = pd.concat([clim.iloc[-10:], clim, clim.iloc[:10]]).rolling(21, center=True, min_periods=3).max().iloc[10:-10]
        env[m] = doy[m].map(dict(zip(range(1, 367), clim.values))).values
    d["cloudfree"] = day & d.n_obs.ge(1) & (kt >= 0.45) & steady & (d.SW_IN <= 1.05 * env)

    # smoke and clean reference steps; partly cloudy steps only train the regression
    d["is_smoke"] = d.cloudfree & (d.AOD550 > 0.2) & (d.AE > 1.0) & (d.hms_smoke == 1)
    d["is_bg"] = d.cloudfree & (d.AOD550 <= 0.15)
    partly = day & (kt >= 0.4) & (d.DF <= d.loc[d.is_smoke, "DF"].quantile(0.98))
    d = d[d.cloudfree | partly]
    if site in maize:
        d = d[d.TIMESTAMP_START.dt.year.isin(cdl.index[cdl[site] == "Corn"])]
    d = d.join(greenness(site), on="date")
    d = d[d.green >= 0.3]  # growing season
    if hemes:
        d = d[(d.NEE_QC == 0) & d.TA.between(20, 30) & (d.VPD < 30)]
    return d[d.GPP.notna()].dropna(subset=drivers)

In [7]:
daily, smoke = [], []
for site in sites:
    d = merged(site)
    s = sample(site, d)
    s[cols].assign(date=s.date.dt.date).round(4).to_csv(data / f"site_{site}.csv", index=False)
    if site == "US-Ha1":
        h = sample(site, d, hemes=False)
        h[cols + ["SW_IN", "cloudfree"]].assign(date=h.date.dt.date).round(4).to_csv(data / "harvard_unfiltered.csv", index=False)
    # figure 1: daily mean aod, and days with smoke steps in the growing season sample without the temperature filter
    a = d.groupby("date").AOD550.mean().dropna()
    days = sample(site, d, hemes=False).query("is_smoke").date.unique()
    daily.append(pd.DataFrame({"site": site, "date": a.index.date, "aod": a.values.round(4)}))
    smoke.append(pd.DataFrame({"site": site, "date": pd.DatetimeIndex(days).date, "aod": a.reindex(days).values.round(4)}))
    print(site, len(s), "steps", int(s.is_smoke.sum()), "smoke", int(s.is_bg.sum()), "clean")
pd.concat(daily).to_csv(data / "aod_daily.csv", index=False)
pd.concat(smoke).to_csv(data / "smoke_days.csv", index=False)

US-Bi2 6524 steps 233 smoke 3032 clean
US-Ne1 3450 steps 214 smoke 504 clean
US-Ne2 1806 steps 112 smoke 256 clean
US-Ne3 2121 steps 162 smoke 304 clean
US-xUN 3883 steps 72 smoke 343 clean
US-UMB 7644 steps 114 smoke 393 clean
US-MMS 5946 steps 74 smoke 319 clean
US-Ha1 6107 steps 89 smoke 519 clean
